# Smoothing_Error_Tradeoff_Seed42

**CPU 런타임**에서 기존 예측 CSV/JSON만 분석한다. 먼저 새 코드를 GitHub에 push한다. 이미지·TAR·체크포인트·학습·GPU 추론은 필요 없다.

- 세 모델의 Baseline / Matched Aug / 기존 teacher–student를 확인한다. 저장된 Soft 및 교사 원본 조건도 발견하면 포함한다. 아직 완료되지 않은 선택 조건은 건너뛴다.
- **고정 판정 기준 0.5의 실제 결과**와 **테스트 정답으로 가능한 판정 기준을 찾아보는 진단**을 구분한다. 진단 수치를 최종 탐지 성능으로 보고하거나 해당 기준을 배포에 사용하지 않는다.
- 같은 조건의 baseline과 비교하여, **FNR을 baseline 이하로 유지하면서 가능한 최소 FPR**을 계산한다. 반대로 FPR을 유지할 때 최소 FNR도 저장한다. 동점 점수 때문에 정확히 같은 오류율을 못 맞출 수 있어 보간 없이 ‘기준 이하’를 사용한다.
- 조건별 기준은 서로 다를 수 있다. 별도로 **네 조건에 하나의 공통 판정 기준을 써도 모든 조건에서 FPR·FNR을 기준 이하로 만들 수 있는지** 확인한다.
- 비교 기준은 Baseline과 Matched Aug 모두 CSV에 저장한다. 마지막 셀의 `REFERENCE`로 표시할 기준을 고른다. AUC 변화도 함께 표시한다. ‘AUC 유지’는 관측 수치 비교이며 통계적 동등성 판정이 아니다.
- 같은 이미지·정답·평가 전처리인지 확인하고, 원래 결과는 수정하지 않는다. 새 결과는 시간별 diagnostics 하위 폴더에 저장한다. 영상 내 프레임 상관을 무시한 유의성 주장은 하지 않는다.

## Cell 1 — Drive와 코드


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys
PROJECT=Path('/content/instagram_deepfake_robustness')
if PROJECT.exists():
    subprocess.run(['git','-C',str(PROJECT),'pull','--ff-only'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/uyh1556/instagram-deepfake-robustness.git',str(PROJECT)],check=True)
assert (PROJECT/'scripts/analyze_smoothing_error_tradeoff.py').is_file(),'새 코드를 먼저 GitHub에 push하세요.'
check=subprocess.run([sys.executable,'-c','import numpy,pandas,sklearn,matplotlib'],capture_output=True,text=True)
if check.returncode:
    print(check.stderr[-1500:])
    subprocess.run([sys.executable,'-m','pip','install','-q','numpy','pandas','scikit-learn','matplotlib'],check=True)
print('CPU 분석 준비 완료. GPU 런타임은 필요 없습니다.')


## Cell 2 — 모델과 기존 결과 경로

새 교사 원본 조건을 Xception에서 아직 실행하지 않았어도 괜찮다. 해당 조건만 제외하고 기존 결과를 분석한다. 새 분석을 다시 하려면 이 셀부터 실행한다.


In [ ]:
from datetime import datetime, timezone
ROOT=Path('/content/drive/MyDrive/instagram-deepfake-robustness')
MODELS=['xception','clip_b16','dinov2_b14']
EVALUATION_ROOTS=[
    ROOT/'evaluations/ffpp_c23_smoothing_mean_teacher_v1/seed42',
    ROOT/'evaluations/ffpp_c23_smoothing_mean_teacher_soft_v1/seed42',
    ROOT/'evaluations/ffpp_c23_smoothing_teacher_views_v1/seed42',
]
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_ROOT=ROOT/'diagnostics/smoothing_error_tradeoff_v1'/f'seed42_{STAMP}'
MODEL_ARGS=' '.join(MODELS)
ROOT_ARGS=' '.join(f'"{p}"' for p in EVALUATION_ROOTS)
for p in EVALUATION_ROOTS:
    print('Found:' if p.is_dir() else 'Optional root absent:',p)
print('Output:',OUTPUT_ROOT)


## Cell 3 — 저장된 예측 분석


In [ ]:
!python -u "{PROJECT}/scripts/analyze_smoothing_error_tradeoff.py" --evaluation-roots {ROOT_ARGS} --output-root "{OUTPUT_ROOT}" --models {MODEL_ARGS} --plots
assert _exit_code==0,'분석 실패: 위 오류 확인. 경로를 수정하거나 Cell 2부터 다시 실행하세요.'


## Cell 4 — 결과 표시

`FPR 감소`가 양수면 기준의 FNR을 넘기지 않으면서 오탐을 줄일 수 있다는 뜻이다. 실제 공통 판정 기준으로 실현된 결과는 아니므로, 아래 공통 기준 표도 함께 본다. `두 오류 모두 감소 가능`은 두 오류가 각각 최소 1장 이상 줄어드는 기준이 존재하는지 나타낸다.


In [ ]:
import csv
from IPython.display import Markdown, Image, display
REFERENCE='baseline'  # 'augmentation'으로 바꾸면 Matched Aug 대비 비교
def read(name):
    with (OUTPUT_ROOT/name).open() as f:
        return list(csv.DictReader(f))
means=read('fixed_condition_means.csv')
points=read('operating_points.csv')
common=read('one_threshold_all_conditions.csv')
labels={'baseline':'Baseline','augmentation':'Matched Aug','mean_teacher':'기존 UDA',
        'mean_teacher_smoothing':'학생 추가 스무딩 UDA','soft_w05':'Soft 0.5','soft_w025':'Soft 0.25',
        'normal_only':'원본 → 원본','clean_to_mixed':'원본 → 혼합'}
conditions={'normal':'원본','smooth_weak':'약','smooth_medium':'중','smooth_strong':'강'}
def yes(value):
    return '가능' if value=='True' else '불가'
for model in MODELS:
    table=[f'### {model} — 실제 고정 기준 결과: 4조건 평균','','|방법|FPR|FNR|BER|AUC|','|---|---:|---:|---:|---:|']
    for r in means:
        if r['model']==model:
            table.append(f"|{labels[r['strategy']]}|{float(r['fpr']):.2%}|{float(r['fnr']):.2%}|{float(r['ber']):.2%}|{float(r['auc']):.4f}|")
    display(Markdown('\n'.join(table)))
    table=[f'### 진단: {labels[REFERENCE]}의 FNR 이하를 유지할 때','','|방법|조건|기준 FNR|달성 FNR|기준 FPR|최소 FPR|FPR 감소 (%p)|ΔAUC|두 오류 모두 감소 가능|',
           '|---|---|---:|---:|---:|---:|---:|---:|---|']
    for r in points:
        if r['model']==model and r['reference']==REFERENCE and r['strategy']!=REFERENCE:
            table.append(f"|{labels[r['strategy']]}|{conditions[r['condition']]}|{float(r['reference_fnr']):.2%}|{float(r['oracle_at_reference_fnr_fnr']):.2%}|{float(r['reference_fpr']):.2%}|{float(r['oracle_at_reference_fnr_fpr']):.2%}|{float(r['oracle_fpr_reduction_pp_at_reference_fnr']):+.2f}|{float(r['delta_auc']):+.4f}|{yes(r['oracle_strictly_lower_both'])}|")
    display(Markdown('\n'.join(table)))
    table=['### 진단: 하나의 공통 판정 기준을 네 조건에 적용','','|방법|네 조건 모두 두 오류 유지·개선 가능|네 조건 모두 두 오류 엄격 감소 가능|네 조건 모두 AUC 유지|',
           '|---|---|---|---|']
    for r in common:
        if r['model']==model and r['reference']==REFERENCE and r['strategy']!=REFERENCE:
            table.append(f"|{labels[r['strategy']]}|{yes(r['oracle_one_threshold_nonworse_every_condition'])}|{yes(r['oracle_one_threshold_strictly_lower_both_every_condition'])}|{'예' if r['auc_maintained_every_condition']=='True' else '아니오'}|")
    display(Markdown('\n'.join(table)))
    figure=OUTPUT_ROOT/f'{model}_error_curves.png'
    if figure.is_file():
        display(Image(filename=str(figure)))
print('테스트 정답을 사용한 가능성 진단입니다. 이 기준을 최종 성능 또는 배포 기준으로 사용하면 안 됩니다.')
print('조건별 가능성이 하나의 공통 기준이나 다른 데이터에서의 성공을 보장하지는 않습니다.')
print('Saved:',OUTPUT_ROOT)


주요 출력: `fixed_condition_means.csv`(실제 성능), `operating_points.csv`(조건별 가능성), `one_threshold_all_conditions.csv`(공통 기준 가능성), `analysis_protocol.json`(입력 해시·누락 조건).
